# Iris Recognition — Full Analysis Notebook

**Task:** Identify persons by iris texture using an ensemble of classical algorithms + a multi-branch CNN.

**Structure:**
1. Dataset exploration & sample visualization
2. Preprocessing (segmentation + rubber-sheet normalization)
3. Part 1 — 4 classical algorithms benchmark
4. Part 1 — Ensemble (Weighted + Stacking)
5. Part 2 — Multi-Branch IrisNet CNN training
6. Part 3 — End-to-end pipeline demo
7. Full comparison table & plots

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2
import torch
from tqdm.notebook import tqdm

from src.dataset import load_dataset, train_test_split_by_person
from src.preprocess import preprocess, segment_iris, rubber_sheet_normalize, load_gray
from src.algorithms import daugman, lbp as lbp_algo, hog_algo, orb_algo
from src.ensemble import WeightedEnsemble, StackingEnsemble
from src.metrics import full_report, compute_eer
from src.neural_model import IrisNet, IrisDataset, train

IRIS_ROOT = '../iris'
TARGET_SIZE = 128

print('All imports OK')

## 1. Dataset Exploration

In [ ]:
paths, labels, classes = load_dataset(IRIS_ROOT)
print(f'Classes ({len(classes)}): {classes}')
print(f'Total images: {len(paths)}')

# Count per class
counts = pd.Series(labels).map({i: c for i, c in enumerate(classes)}).value_counts()
fig, ax = plt.subplots(figsize=(12, 4))
counts.sort_index().plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Images per Person', fontsize=14, fontweight='bold')
ax.set_xlabel('Person')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Show sample raw images
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, cls in enumerate(classes[:16]):
    cls_paths = [p for p, l in zip(paths, labels) if l == i]
    img = cv2.imread(cls_paths[0])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax = axes[i // 8][i % 8]
    ax.imshow(img)
    ax.set_title(cls[:8], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Raw Eye Images (1 per person)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Preprocessing — Segmentation & Normalization

In [ ]:
# Visualize full preprocessing pipeline on 4 samples
sample_paths = [paths[i * 80] for i in range(4)]

fig, axes = plt.subplots(4, 3, figsize=(12, 12))
col_titles = ['Raw (gray)', 'Segmented', 'Normalized (rubber-sheet)']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=11, fontweight='bold')

for row, path in enumerate(sample_paths):
    gray = load_gray(path)
    pupil, iris_circle = segment_iris(gray)
    norm = preprocess(path, target_size=TARGET_SIZE)
    
    # Draw circles on copy
    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    cv2.circle(vis, (pupil[0], pupil[1]), pupil[2], (0, 0, 255), 2)
    cv2.circle(vis, (iris_circle[0], iris_circle[1]), iris_circle[2], (0, 255, 0), 2)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    
    axes[row][0].imshow(gray, cmap='gray')
    axes[row][1].imshow(vis_rgb)
    axes[row][2].imshow(norm, cmap='gray', aspect='auto')
    for ax in axes[row]:
        ax.axis('off')
    name = classes[labels[row * 80]]
    axes[row][0].set_ylabel(name, fontsize=9)

plt.suptitle('Preprocessing Pipeline\nRed=pupil, Green=iris boundary', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Part 1 — Classical Algorithm Benchmark

In [ ]:
# Train/test split
train_paths, train_labels, test_paths, test_labels = train_test_split_by_person(
    paths, labels, classes, test_ratio=0.2
)
print(f'Train: {len(train_paths)}, Test: {len(test_paths)}')

ALGORITHMS = {
    'Daugman': {'extract': daugman.extract, 'similarity': daugman.similarity},
    'LBP':     {'extract': lbp_algo.extract, 'similarity': lbp_algo.similarity},
    'HOG':     {'extract': hog_algo.extract, 'similarity': hog_algo.similarity},
    'ORB':     {'extract': orb_algo.extract, 'similarity': orb_algo.similarity},
}

In [ ]:
# Preprocess all images
def preprocess_batch(paths):
    norms, valid_labels = [], []
    for path, label in tqdm(zip(paths, train_labels if paths is train_paths else test_labels),
                             total=len(paths)):
        try:
            norms.append(preprocess(path, target_size=TARGET_SIZE))
            valid_labels.append(label)
        except:
            pass
    return norms, valid_labels

print('Preprocessing train...')
train_norms, train_labels_v = preprocess_batch(train_paths)
print('Preprocessing test...')
test_norms, test_labels_v = preprocess_batch(test_paths)
print(f'Train norms: {len(train_norms)}, Test norms: {len(test_norms)}')

In [ ]:
# Extract features for each algorithm
train_feats = {name: [] for name in ALGORITHMS}
test_feats  = {name: [] for name in ALGORITHMS}

for name, algo in ALGORITHMS.items():
    print(f'Extracting [{name}]...')
    for norm in tqdm(train_norms):
        train_feats[name].append(algo['extract'](norm))
    for norm in tqdm(test_norms):
        test_feats[name].append(algo['extract'](norm))
print('Done.')

In [ ]:
# Benchmark each algorithm
import random

def nn_identify(train_f, train_l, test_f, sim_fn):
    preds = []
    for qf in test_f:
        best_s, best_l = -1, -1
        for gf, gl in zip(train_f, train_l):
            s = sim_fn(qf, gf)
            if s > best_s:
                best_s, best_l = s, gl
        preds.append(best_l)
    return np.array(preds)

def build_scores(feats_by_class, sim_fn, seed=42):
    rng = random.Random(seed)
    genuine, impostor = [], []
    clss = list(feats_by_class.keys())
    for cls, feats in feats_by_class.items():
        if len(feats) < 2: continue
        for i in range(len(feats)-1):
            genuine.append(sim_fn(feats[i], feats[i+1]))
        others = [c for c in clss if c != cls]
        if not others: continue
        for _ in range(len(feats)-1):
            oc = rng.choice(others)
            of = rng.choice(feats_by_class[oc])
            impostor.append(sim_fn(feats[rng.randint(0,len(feats)-1)], of))
    return np.array(genuine, dtype=np.float32), np.array(impostor, dtype=np.float32)

reports = []
algo_genuine, algo_impostor = {}, {}

for name, algo in ALGORITHMS.items():
    print(f'\n[{name}]')
    sim_fn = algo['similarity']
    
    preds = nn_identify(train_feats[name], train_labels_v,
                         test_feats[name], sim_fn)
    
    test_by_class = {}
    for f, l in zip(test_feats[name], test_labels_v):
        test_by_class.setdefault(l, []).append(f)
    
    genuine, impostor = build_scores(test_by_class, sim_fn)
    algo_genuine[name] = genuine
    algo_impostor[name] = impostor
    
    r = full_report(name, genuine, impostor,
                    y_true=np.array(test_labels_v), y_pred=preds)
    reports.append(r)
    print(f'  EER={r["EER"]:.4f}  AUC={r["AUC"]:.4f}  Acc={r.get("Accuracy")}')

## 4. Part 1 — Ensemble

In [ ]:
from src.metrics import auc_score

algo_names = list(ALGORITHMS.keys())
min_gen = min(len(algo_genuine[n]) for n in algo_names)
min_imp = min(len(algo_impostor[n]) for n in algo_names)

gen_matrix  = np.stack([algo_genuine[n][:min_gen]  for n in algo_names], axis=1)
imp_matrix  = np.stack([algo_impostor[n][:min_imp] for n in algo_names], axis=1)
score_matrix = np.vstack([gen_matrix, imp_matrix])
ens_labels   = np.concatenate([np.ones(min_gen), np.zeros(min_imp)]).astype(int)

# Weighted ensemble
w_ens = WeightedEnsemble()
w_ens.optimize_weights(score_matrix, ens_labels)
ens_s = score_matrix @ w_ens.weights
ens_gen  = ens_s[ens_labels == 1]
ens_imp  = ens_s[ens_labels == 0]
ens_eer, _ = compute_eer(ens_gen, ens_imp)

# Stacking ensemble
stack_ens = StackingEnsemble()
stack_ens.fit(score_matrix, ens_labels)
stack_p = np.array([stack_ens.predict_proba(r) for r in score_matrix])
stack_gen  = stack_p[ens_labels == 1]
stack_imp  = stack_p[ens_labels == 0]
stack_eer, _ = compute_eer(stack_gen, stack_imp)

reports.append({'algorithm': 'Ensemble (Weighted)', 'EER': round(float(ens_eer),4),
                'AUC': round(auc_score(ens_gen, ens_imp),4)})
reports.append({'algorithm': 'Ensemble (Stacking)', 'EER': round(float(stack_eer),4),
                'AUC': round(auc_score(stack_gen, stack_imp),4)})

print(f'Weighted Ensemble  EER={ens_eer:.4f}')
print(f'Stacking Ensemble  EER={stack_eer:.4f}')
print(f'Optimized weights: {dict(zip(algo_names, w_ens.weights.round(3)))}')

## 5. Part 2 — IrisNet CNN Training

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

train_ds = IrisDataset(train_norms, train_labels_v, augment=True)
val_ds   = IrisDataset(test_norms,  test_labels_v,  augment=False)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=16, shuffle=False)

model = IrisNet(n_classes=len(classes), branch_dim=128, dropout=0.4)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
history = train(model, train_loader, val_loader, n_epochs=30, lr=1e-3, device=device)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['train_loss'])+1)
ax1.plot(epochs, history['train_loss'], label='Train')
ax1.plot(epochs, history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(epochs, history['train_acc'], label='Train')
ax2.plot(epochs, history['val_acc'],   label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.suptitle('IrisNet Training History', fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Final val accuracy: {history["val_acc"][-1]:.4f}')

## 6. Full Comparison Table

In [ ]:
# Add CNN result
reports.append({'algorithm': 'IrisNet CNN',
                'Accuracy': round(history['val_acc'][-1], 4)})

df = pd.DataFrame(reports)
print(df.to_string(index=False))

In [ ]:
# Comparison bar charts
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0','#00BCD4','#795548']

for ax, metric in zip(axes, ['EER', 'AUC', 'Accuracy']):
    sub = df.dropna(subset=[metric])
    ax.bar(sub['algorithm'], sub[metric],
           color=colors[:len(sub)], edgecolor='white')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=35)
    ax.set_ylim(0, 1)
    for bar, val in zip(ax.patches, sub[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Iris Recognition — Algorithm Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('../iris_recognition/results/figures', exist_ok=True)
plt.savefig('../iris_recognition/results/figures/comparison.png', dpi=150, bbox_inches='tight')
plt.show()